In [1]:
# ============================================================================
# NOTEBOOK 05 — ETL DE LA DEMANDA ELÉCTRICA (Pandas)
# ============================================================================
# Objetivo: Procesar los 40 JSON descargados de REData en un único dataset
# limpio y enriquecido con features de calendario, listo para el LSTM.
#
# ============================================================================

import os
import json
import math
import shutil
import pandas as pd
import numpy as np

RUTA_RAW = "/opt/spark-data/raw/redata"
RUTA_PROCESSED = "/opt/spark-data/processed/demanda_clean.parquet"
RUTA_CSV = "/opt/spark-data/features/demanda_clean.csv"

print(f"Raw:       {RUTA_RAW}")
print(f"Processed: {RUTA_PROCESSED}")
print(f"CSV:       {RUTA_CSV}")

Raw:       /opt/spark-data/raw/redata
Processed: /opt/spark-data/processed/demanda_clean.parquet
CSV:       /opt/spark-data/features/demanda_clean.csv


In [2]:
# Leemos todos los JSON y extraemos los puntos horarios al DataFrame
registros = []

for f in sorted(os.listdir(RUTA_RAW)):
    if not f.endswith(".json"):
        continue
    with open(os.path.join(RUTA_RAW, f)) as fp:
        data = json.load(fp)
    for punto in data['included'][0]['attributes']['values']:
        registros.append({
            'datetime_str': punto['datetime'],
            'demanda_mw':   float(punto['value']),
        })

df = pd.DataFrame(registros)
print(f"Total puntos cargados: {len(df):,}")
df.head()

Total puntos cargados: 29,183


,datetime_str,demanda_mw
0,2023-01-01T00:00:00.000+01:00,19872.637
1,2023-01-01T01:00:00.000+01:00,19198.718
2,2023-01-01T02:00:00.000+01:00,18099.597
3,2023-01-01T03:00:00.000+01:00,17142.238
4,2023-01-01T04:00:00.000+01:00,16539.586


In [3]:
# REData devuelve timestamps con zona ("+01:00", "+02:00").
# Los convertimos a UTC para homogeneidad y evitar problemas con cambios de hora.

df['timestamp'] = pd.to_datetime(df['datetime_str'], utc=True)
df = df[['timestamp', 'demanda_mw']].sort_values('timestamp').reset_index(drop=True)

print(f"Rango: {df['timestamp'].min()}  →  {df['timestamp'].max()}")
print(f"Filas: {len(df):,}")

Rango: 2022-12-31 23:00:00+00:00  →  2026-04-30 21:00:00+00:00
Filas: 29,183


In [ ]:
# Generamos la secuencia teórica (uno por hora) y comprobamos huecos.

df_esperado = pd.DataFrame({
    'timestamp': pd.date_range(df['timestamp'].min(), df['timestamp'].max(), freq='1h', tz='UTC')
})

df_completo = df_esperado.merge(df, on='timestamp', how='left')
n_huecos = df_completo['demanda_mw'].isnull().sum()
print(f"Timestamps esperados: {len(df_esperado):,}")
print(f"Huecos detectados:    {n_huecos}")

if n_huecos > 0:
    df_completo['demanda_mw'] = df_completo['demanda_mw'].interpolate(method='linear')
    print(f"✓ Huecos imputados por interpolación lineal")

df = df_completo
assert df['demanda_mw'].isnull().sum() == 0, "Quedan nulos tras la imputación"

Timestamps esperados: 29,183
Huecos detectados:    0


In [ ]:
# Trabajamos con timestamp convertido a hora local para que hora/día/mes tengan sentido humano (no UTC).

df['timestamp_local'] = df['timestamp'].dt.tz_convert('Europe/Madrid')

df['hora']     = df['timestamp_local'].dt.hour
df['dia_sem']  = df['timestamp_local'].dt.dayofweek    # 0=lunes, 6=domingo
df['mes']      = df['timestamp_local'].dt.month
df['dia_anio'] = df['timestamp_local'].dt.dayofyear

# Codificación cíclica con sin/cos para que el LSTM aprenda la naturaleza periódica
df['hora_sin'] = np.sin(df['hora'] * 2 * math.pi / 24)
df['hora_cos'] = np.cos(df['hora'] * 2 * math.pi / 24)
df['dia_sin']  = np.sin(df['dia_sem'] * 2 * math.pi / 7)
df['dia_cos']  = np.cos(df['dia_sem'] * 2 * math.pi / 7)
df['mes_sin']  = np.sin((df['mes']-1) * 2 * math.pi / 12)
df['mes_cos']  = np.cos((df['mes']-1) * 2 * math.pi / 12)

df['es_finde'] = (df['dia_sem'] >= 5).astype(int)

print(df[['timestamp_local', 'hora', 'dia_sem', 'mes', 'es_finde']].head())

            timestamp_local  hora  dia_sem  mes  es_finde
0 2023-01-01 00:00:00+01:00     0        6    1         1
1 2023-01-01 01:00:00+01:00     1        6    1         1
2 2023-01-01 02:00:00+01:00     2        6    1         1
3 2023-01-01 03:00:00+01:00     3        6    1         1
4 2023-01-01 04:00:00+01:00     4        6    1         1


In [6]:
# Festivos nacionales fijos + Jueves y Viernes Santo de cada año
festivos = []
for anio in range(2023, 2027):
    festivos.extend([
        f"{anio}-01-01",  # Año Nuevo
        f"{anio}-01-06",  # Reyes
        f"{anio}-05-01",  # Día del Trabajo
        f"{anio}-08-15",  # Asunción
        f"{anio}-10-12",  # Hispanidad
        f"{anio}-11-01",  # Todos los Santos
        f"{anio}-12-06",  # Constitución
        f"{anio}-12-08",  # Inmaculada
        f"{anio}-12-25",  # Navidad
    ])

# Semana Santa (Jueves y Viernes Santo)
festivos.extend([
    "2023-04-06", "2023-04-07",
    "2024-03-28", "2024-03-29",
    "2025-04-17", "2025-04-18",
    "2026-04-02", "2026-04-03",
])

festivos_set = set(pd.to_datetime(festivos).date)
df['es_festivo'] = df['timestamp_local'].dt.date.map(lambda x: 1 if x in festivos_set else 0)

n_festivos = df['es_festivo'].sum()
print(f"Horas marcadas como festivo: {n_festivos:,} (= {n_festivos // 24} días completos)")

Horas marcadas como festivo: 888 (= 37 días completos)


In [7]:
# Seleccionamos columnas finales en orden lógico. Eliminamos zona horaria 
# del timestamp para que Parquet y CSV se serialicen sin problemas.

df_final = df[[
    'timestamp',
    'demanda_mw',
    'hora', 'dia_sem', 'mes', 'dia_anio',
    'hora_sin', 'hora_cos',
    'dia_sin', 'dia_cos',
    'mes_sin', 'mes_cos',
    'es_finde', 'es_festivo',
]].copy()

df_final['timestamp'] = df_final['timestamp'].dt.tz_localize(None)

print(f"Shape: {df_final.shape}")
print(f"\nEstadísticas iniciales de la demanda:")
print(df_final['demanda_mw'].describe())

Shape: (29183, 14)

Estadísticas iniciales de la demanda:
count    29183.000000
mean     26923.173584
std       4330.011139
min        518.113000
25%      23396.493500
50%      26960.301000
75%      29985.453000
max      41725.907000
Name: demanda_mw, dtype: float64


In [8]:
# El mínimo histórico realista del sistema peninsular ronda los 16.000-18.000 MW.
# Inspeccionamos si hay valores anómalos que requieran tratamiento.

print("10 valores más bajos:")
print(df_final.nsmallest(10, 'demanda_mw')[['timestamp', 'demanda_mw']].to_string(index=False))

print("\nDistribución de valores bajos:")
for umbral in [5000, 10000, 14000, 16000, 18000]:
    n = (df_final['demanda_mw'] < umbral).sum()
    print(f"  Demanda < {umbral:>5} MW: {n:>4} puntos")

10 valores más bajos:
          timestamp  demanda_mw
2025-04-28 11:00:00     518.113
2025-04-28 12:00:00    1172.419
2025-04-28 13:00:00    2022.939
2025-04-28 14:00:00    2764.712
2025-04-28 15:00:00    3861.053
2025-04-28 16:00:00    5811.180
2025-04-28 17:00:00    7343.159
2025-04-28 18:00:00    9495.675
2025-04-28 19:00:00   11775.423
2025-04-28 20:00:00   13157.059

Distribución de valores bajos:
  Demanda <  5000 MW:    5 puntos
  Demanda < 10000 MW:    8 puntos
  Demanda < 14000 MW:   11 puntos
  Demanda < 16000 MW:   16 puntos
  Demanda < 18000 MW:   70 puntos


In [9]:
# CONTEXTO TÉCNICO:
# El 28 de abril de 2025, a las 12:33 hora peninsular, se produjo un cero técnico 
# del sistema eléctrico ibérico (apagón total). El dataset muestra demanda anómala:
#   - 28-abr 09:00 → 23:00 UTC: caída + cero técnico + reposición inicial (15 h)
#   - 29-abr 00:00 → 03:00 UTC: reposición progresiva en madrugada (4 h)
# Total: 19 horas no representativas del sistema en condiciones normales.
#
# El resto de mínimos detectados (madrugadas de Año Nuevo ~16.300 MW) son valores
# REALES coherentes con el patrón anual del sistema y se conservan sin modificar.
#
# ESTRATEGIA: para cada hora afectada, imputamos con el promedio de la misma 
# hora del mismo día-semana en las 2 semanas previas y 2 posteriores.

inicio_apagon = pd.Timestamp("2025-04-28 09:00:00")
fin_apagon = pd.Timestamp("2025-04-29 03:00:00")
mask_apagon = (df_final['timestamp'] >= inicio_apagon) & (df_final['timestamp'] <= fin_apagon)

horas_apagon = df_final.loc[mask_apagon, 'timestamp'].tolist()
valores_originales = df_final.loc[mask_apagon, 'demanda_mw'].copy().tolist()

valores_imputados = []
for ts in horas_apagon:
    referencias = []
    for delta in [-2, -1, 1, 2]:
        ts_ref = ts + pd.Timedelta(weeks=delta)
        match = df_final[df_final['timestamp'] == ts_ref]
        if not match.empty:
            referencias.append(match['demanda_mw'].iloc[0])
    valores_imputados.append(np.mean(referencias) if referencias else None)

df_final.loc[mask_apagon, 'demanda_mw'] = valores_imputados

print(f"Horas imputadas: {len(horas_apagon)}")
print("\nDetalle (antes → después):")
for ts, antes, despues in zip(horas_apagon, valores_originales, valores_imputados):
    print(f"  {ts}:  {antes:>8.0f}  →  {despues:>8.0f}  MW")

Horas imputadas: 19

Detalle (antes → después):
  2025-04-28 09:00:00:     25690  →     26574  MW
  2025-04-28 10:00:00:     13999  →     26564  MW
  2025-04-28 11:00:00:       518  →     26637  MW
  2025-04-28 12:00:00:      1172  →     26210  MW
  2025-04-28 13:00:00:      2023  →     25572  MW
  2025-04-28 14:00:00:      2765  →     25215  MW
  2025-04-28 15:00:00:      3861  →     25431  MW
  2025-04-28 16:00:00:      5811  →     25988  MW
  2025-04-28 17:00:00:      7343  →     27315  MW
  2025-04-28 18:00:00:      9496  →     29097  MW
  2025-04-28 19:00:00:     11775  →     30544  MW
  2025-04-28 20:00:00:     13157  →     28268  MW
  2025-04-28 21:00:00:     14034  →     25531  MW
  2025-04-28 22:00:00:     14585  →     23640  MW
  2025-04-28 23:00:00:     15155  →     22291  MW
  2025-04-29 00:00:00:     15405  →     21382  MW
  2025-04-29 01:00:00:     15799  →     20982  MW
  2025-04-29 02:00:00:     16244  →     20899  MW
  2025-04-29 03:00:00:     17151  →     21519  MW


In [10]:
print("Estadísticas finales de la demanda:")
print(df_final['demanda_mw'].describe())

print("\n5 valores más bajos tras la imputación:")
print(df_final.nsmallest(5, 'demanda_mw')[['timestamp', 'demanda_mw']].to_string(index=False))
print("\n(Esperado: madrugadas de Año Nuevo y festivos, valores reales del sistema)")

Estadísticas finales de la demanda:
count    29183.000000
mean     26932.551444
std       4308.004255
min      16272.179000
25%      23402.813000
50%      26960.931000
75%      29985.919000
max      41725.907000
Name: demanda_mw, dtype: float64

5 valores más bajos tras la imputación:
          timestamp  demanda_mw
2023-01-01 04:00:00   16272.179
2023-01-01 05:00:00   16324.629
2023-01-01 07:00:00   16510.280
2023-01-01 03:00:00   16539.586
2023-01-01 06:00:00   16582.757

(Esperado: madrugadas de Año Nuevo y festivos, valores reales del sistema)


In [11]:
if os.path.isdir(RUTA_PROCESSED):
    shutil.rmtree(RUTA_PROCESSED)
os.makedirs(os.path.dirname(RUTA_PROCESSED), exist_ok=True)

df_final.to_parquet(RUTA_PROCESSED, engine='pyarrow', compression='snappy', index=False)

size_mb = os.path.getsize(RUTA_PROCESSED) / 1024**2
print(f"✓ Parquet guardado: {RUTA_PROCESSED}")
print(f"✓ Tamaño: {size_mb:.2f} MB")

✓ Parquet guardado: /opt/spark-data/processed/demanda_clean.parquet
✓ Tamaño: 0.49 MB


In [12]:
os.makedirs(os.path.dirname(RUTA_CSV), exist_ok=True)
df_final.to_csv(RUTA_CSV, index=False)

size_mb = os.path.getsize(RUTA_CSV) / 1024**2
print(f"✓ CSV guardado: {RUTA_CSV}")
print(f"✓ Tamaño: {size_mb:.2f} MB")

✓ CSV guardado: /opt/spark-data/features/demanda_clean.csv
✓ Tamaño: 4.14 MB


In [14]:
print("="*60)
print("RESUMEN ETL DEMANDA REData")
print("="*60)
print(f"Herramienta:       Pandas (volumen acotado: ~29K registros)")
print(f"Origen (40 JSON):  {RUTA_RAW}")
print(f"Parquet:           {RUTA_PROCESSED}")
print(f"CSV para LSTM:     {RUTA_CSV}")
print()
print(f"Filas finales:     {len(df_final):,}")
print(f"Cobertura:         {df_final['timestamp'].min()}  →  {df_final['timestamp'].max()}")
print(f"Columnas:          {df_final.shape[1]} (timestamp + demanda + 12 features calendario)")
print()
print("LIMPIEZA APLICADA:")
print(f"  - Conversión a UTC para serie continua sin cambios de hora")
print(f"  - Imputación de huecos por interpolación lineal")
print(f"  - Imputación de 19 horas del apagón ibérico (28-29 abril 2025)")
print(f"    Estrategia: media de misma hora en ±2 semanas")
print()
print("DEMANDA FINAL (MW):")
print(f"  Mínimo:    {df_final['demanda_mw'].min():>7.0f}  (madrugada de Año Nuevo)")
print(f"  Media:     {df_final['demanda_mw'].mean():>7.0f}")
print(f"  Máximo:    {df_final['demanda_mw'].max():>7.0f}")
print(f"  Desv.std:  {df_final['demanda_mw'].std():>7.0f}")
print("="*60)

RESUMEN ETL DEMANDA REData
Herramienta:       Pandas (volumen acotado: ~29K registros)
Origen (40 JSON):  /opt/spark-data/raw/redata
Parquet:           /opt/spark-data/processed/demanda_clean.parquet
CSV para LSTM:     /opt/spark-data/features/demanda_clean.csv

Filas finales:     29,183
Cobertura:         2022-12-31 23:00:00  →  2026-04-30 21:00:00
Columnas:          14 (timestamp + demanda + 12 features calendario)

LIMPIEZA APLICADA:
  - Conversión a UTC para serie continua sin cambios de hora
  - Imputación de huecos por interpolación lineal
  - Imputación de 19 horas del apagón ibérico (28-29 abril 2025)
    Estrategia: media de misma hora en ±2 semanas

DEMANDA FINAL (MW):
  Mínimo:      16272  (madrugada de Año Nuevo)
  Media:       26933
  Máximo:      41726
  Desv.std:     4308
